# 03 - Síntesis de arquitecturas tabulares profundas

**Comparación conjunta de los experimentos de clasificación y regresión**

Este cuaderno integra los resultados persistidos por `01_classification.ipynb` y `02_regression.ipynb`. No vuelve a entrenar modelos ni abre nuevas decisiones de selección: verifica los artefactos, resume variabilidad entre semillas y estudia qué comportamiento arquitectónico se conserva o cambia entre Airlines y California Housing.

## 1. Objetivo y alcance

La pregunta central no es qué valor numérico es mayor entre tareas, porque ROC-AUC y RMSE miden objetos distintos. El objetivo es comparar cada modelo dentro de su propio problema y luego alinear evidencias compatibles: posición relativa, variabilidad entre semillas, costo, tamaño y dependencia del tipo de variables.

El baseline lineal se conserva únicamente como referencia interna de cada tarea. La comparación principal sigue centrada en TabNet, TabTransformer, FT-Transformer y SAINT supervisado.

## 2. Reglas de interpretación

1. Clasificación se ordena por ROC-AUC; balanced accuracy, F1 y PR-AUC verifican que el ranking no dependa de una sola lectura.
2. Regresión se ordena por RMSE; MAE, mediana del error absoluto y R² describen magnitud típica, extremos y ajuste global.
3. Las tres semillas comparten el mismo split dentro de cada tarea. Su desviación estándar mide sensibilidad a inicialización y minibatches, no incertidumbre poblacional completa.
4. Las diferencias por semilla son pareadas, pero con tres pares no se justifican pruebas de significancia asintóticas.
5. Tiempos de clasificación y regresión no se comparan directamente: cambian dataset, número de filas y presupuesto de épocas.
6. Este análisis es post hoc sobre test. Sus observaciones no deben utilizarse para reajustar modelos y volver a evaluar el mismo test.

## 3. Importaciones

La carga, validación y agregación permanecen en `src/evaluation.py`. El notebook contiene únicamente configuración, llamadas de alto nivel y presentación.

In [1]:
from pathlib import Path
import logging

import pandas as pd
from IPython.display import display

from src.evaluation import (
    build_cross_task_architecture_table,
    build_seed_rank_table,
    load_persisted_task_results,
    paired_seed_differences,
    plot_cross_task_performance,
    plot_cross_task_resources,
    plot_seed_rank_consistency,
    summarize_task_metrics,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
pd.set_option("display.max_columns", 120)

## 4. Configuración de campañas

Los identificadores siguientes son contratos, no filtros informales. Una campaña solo se acepta si contiene exactamente los modelos y semillas esperados, una única huella de protocolo, un único split, predicciones fila a fila y las implementaciones declaradas. Resultados históricos incompatibles se ignoran de forma explícita.

In [2]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError(
        "Run this notebook from the repository root."
    )

FIGURES_DIR = PROJECT_ROOT / "results" / "figures"
EXPECTED_SEEDS = (42, 123, 2025)

COMPARISON_CONFIG = {
    "classification": {
        "results_dir": PROJECT_ROOT / "results" / "classification",
        "experiment_id": "airlines_dl_architectures_v2",
        "models": (
            "baseline_logistic",
            "tabnet",
            "tab_transformer",
            "ft_transformer",
            "saint_supervised",
        ),
        "implementations": {
            "baseline_logistic": "sklearn_logistic_regression",
            "tabnet": "pytorch_tabnet_4_1",
            "tab_transformer": "tab_transformer_v1",
            "ft_transformer": "ft_transformer_v1",
            "saint_supervised": "saint_column_v1",
        },
    },
    "regression": {
        "results_dir": PROJECT_ROOT / "results" / "regression",
        "experiment_id": "california_housing_dl_architectures_v2",
        "models": (
            "baseline_ridge",
            "tabnet",
            "tab_transformer",
            "ft_transformer",
            "saint_supervised",
        ),
        "implementations": {
            "baseline_ridge": "sklearn_ridge",
            "tabnet": "pytorch_tabnet_4_1",
            "tab_transformer": "tab_transformer_v1",
            "ft_transformer": "ft_transformer_v1",
            "saint_supervised": "saint_column_v1",
        },
    },
}

## 5. Carga y auditoría de artefactos

La auditoría no confía solo en `summary_metrics.csv`. Para cada ejecución abre configuración, historial y predicciones; verifica checkpoint, recarga certificada, índices de test, probabilidades o residuos, y recalcula las métricas desde las predicciones persistidas.

In [3]:
classification_config = COMPARISON_CONFIG["classification"]
regression_config = COMPARISON_CONFIG["regression"]

classification_artifacts = load_persisted_task_results(
    results_dir=classification_config["results_dir"],
    task="classification",
    experiment_id=classification_config["experiment_id"],
    expected_models=classification_config["models"],
    expected_seeds=EXPECTED_SEEDS,
    expected_implementations=classification_config["implementations"],
)
regression_artifacts = load_persisted_task_results(
    results_dir=regression_config["results_dir"],
    task="regression",
    experiment_id=regression_config["experiment_id"],
    expected_models=regression_config["models"],
    expected_seeds=EXPECTED_SEEDS,
    expected_implementations=regression_config["implementations"],
)
COMPARISON_READY = classification_artifacts.ready and regression_artifacts.ready

## 6. Estado de integridad

`ready=True` exige una campaña completa. Si alguna tarea aparece pendiente, las secciones disponibles siguen siendo ejecutables, pero la síntesis cruzada permanece desactivada para evitar conclusiones basadas en campañas distintas.

In [4]:
readiness_table = pd.DataFrame(
    [
        {
            "task": artifacts.task,
            "ready": artifacts.ready,
            "experiment_id": artifacts.experiment_id,
            "loaded_runs": artifacts.report["loaded_runs"],
            "expected_runs": artifacts.report["expected_runs"],
            "issues": len(artifacts.report["issues"]),
        }
        for artifacts in (classification_artifacts, regression_artifacts)
    ]
)
display(readiness_table)

campaign_contract_rows = []
for artifacts in (classification_artifacts, regression_artifacts):
    if not artifacts.ready:
        continue
    first_config = next(iter(artifacts.configs.values()))
    first_prediction = next(iter(artifacts.predictions.values()))
    campaign_contract_rows.append(
        {
            "task": artifacts.task,
            "dataset": first_config["data_metadata"]["dataset_name"],
            "device": first_config["experiment"]["device"],
            "protocol_fingerprint": artifacts.metrics[
                "protocol_fingerprint"
            ].iloc[0],
            "split_fingerprint": artifacts.metrics[
                "split_fingerprint"
            ].iloc[0],
            "test_rows": len(first_prediction),
            "seeds": tuple(sorted(artifacts.metrics["seed"].unique())),
            "prediction_scope": artifacts.metrics[
                "prediction_scope"
            ].unique().item(),
        }
    )
if campaign_contract_rows:
    display(pd.DataFrame(campaign_contract_rows))

for artifacts in (classification_artifacts, regression_artifacts):
    if artifacts.report["issues"]:
        display(
            pd.Series(
                artifacts.report["issues"],
                name=f"{artifacts.task}_integrity_issues",
            )
        )

,task,ready,experiment_id,loaded_runs,expected_runs,issues
0,classification,True,airlines_dl_architectures_v2,15,15,0
1,regression,True,california_housing_dl_architectures_v2,15,15,0


,task,dataset,device,protocol_fingerprint,split_fingerprint,test_rows,seeds,prediction_scope
0,classification,airlines_passenger_satisfaction,cuda,7579154fb690ea44132fe95bd0c19decaa106639acd541...,7ecf8a1d36683939f63a67274a56018c44f452d24070ba...,19482,"(42, 123, 2025)",row_independent
1,regression,california_housing,cuda,dd54cb529e3297af8bd178329af49141402cad6fa73c38...,56032023365263ab402e114e4d1cb550b614272be8aae4...,3096,"(42, 123, 2025)",row_independent


## 7. Clasificación: rendimiento entre semillas

ROC-AUC evalúa ordenamiento probabilístico sin fijar un único umbral. Balanced accuracy y F1 complementan esa lectura: la primera pondera por igual ambas clases y la segunda resume precision y recall de la clase positiva. La desviación estándar se interpreta junto con la media; elegir la mejor semilla sería optimismo de selección.

In [5]:
classification_summary = (
    summarize_task_metrics(classification_artifacts.metrics, "classification")
    if classification_artifacts.ready
    else pd.DataFrame()
)
if classification_artifacts.ready:
    classification_columns = [
        "model_name",
        "roc_auc_mean",
        "roc_auc_std",
        "balanced_accuracy_mean",
        "f1_mean",
        "pr_auc_mean",
        "within_task_rank",
        "primary_effect_vs_baseline",
    ]
    display(classification_summary[classification_columns])
else:
    display(pd.Series({"status": "classification campaign pending"}))

,model_name,roc_auc_mean,roc_auc_std,balanced_accuracy_mean,f1_mean,pr_auc_mean,within_task_rank,primary_effect_vs_baseline
0,saint_supervised,0.995818,1.851853e-04,0.964595,0.961591,0.995020,1,0.068148
1,ft_transformer,0.995298,9.246328e-05,0.963153,0.959763,0.994412,2,0.067629
2,tab_transformer,0.994777,1.519148e-04,0.960971,0.956963,0.993857,3,0.067108
3,tabnet,0.989872,1.842124e-04,0.948178,0.942665,0.988117,4,0.062202
4,baseline_logistic,0.927669,1.359740e-16,0.869256,0.851742,0.931665,5,0.000000


## 8. Regresión: rendimiento entre semillas

RMSE penaliza con mayor intensidad los errores grandes; MAE describe el error absoluto promedio y la mediana reduce la influencia de casos extremos. R² contextualiza la reducción de varianza residual. Ninguna de estas métricas se compara numéricamente con ROC-AUC.

In [6]:
regression_summary = (
    summarize_task_metrics(regression_artifacts.metrics, "regression")
    if regression_artifacts.ready
    else pd.DataFrame()
)
if regression_artifacts.ready:
    regression_columns = [
        "model_name",
        "rmse_mean",
        "rmse_std",
        "mae_mean",
        "median_absolute_error_mean",
        "r2_mean",
        "within_task_rank",
        "primary_effect_vs_baseline",
    ]
    display(regression_summary[regression_columns])
else:
    display(pd.Series({"status": "regression campaign pending"}))

,model_name,rmse_mean,rmse_std,mae_mean,median_absolute_error_mean,r2_mean,within_task_rank,primary_effect_vs_baseline
0,saint_supervised,0.485111,0.003245,0.330029,0.222751,0.822024,1,32.741007
1,ft_transformer,0.501123,0.006163,0.340173,0.225641,0.810067,2,30.520929
2,tab_transformer,0.515159,0.004412,0.357733,0.250330,0.799289,3,28.574905
3,tabnet,0.676694,0.025412,0.461130,0.331366,0.653375,4,6.178610
4,baseline_ridge,0.721258,0.000000,0.521037,0.404699,0.606588,5,0.000000


## 9. Estabilidad del orden por semilla

Las semillas están emparejadas entre modelos. Se comprueba si el orden cambia al variar inicialización y minibatches. La tabla de diferencias orienta el signo a favor del mejor modelo promedio: un valor positivo indica ventaja de la referencia. Con solo tres semillas se reportan magnitud y consistencia, no p-values.

In [7]:
classification_ranks = (
    build_seed_rank_table(classification_artifacts.metrics, "classification")
    if classification_artifacts.ready
    else pd.DataFrame()
)
regression_ranks = (
    build_seed_rank_table(regression_artifacts.metrics, "regression")
    if regression_artifacts.ready
    else pd.DataFrame()
)
classification_differences = (
    paired_seed_differences(
        classification_artifacts.metrics,
        "classification",
    )
    if classification_artifacts.ready
    else pd.DataFrame()
)
regression_differences = (
    paired_seed_differences(regression_artifacts.metrics, "regression")
    if regression_artifacts.ready
    else pd.DataFrame()
)
if classification_artifacts.ready:
    display(classification_differences)
if regression_artifacts.ready:
    display(regression_differences)

,reference_model,competitor_model,metric,n_paired_seeds,mean_reference_advantage,std_reference_advantage,reference_wins,ties
0,saint_supervised,baseline_logistic,roc_auc,3,0.068148,0.000185,3,0
1,saint_supervised,tabnet,roc_auc,3,0.005946,0.000369,3,0
2,saint_supervised,tab_transformer,roc_auc,3,0.001041,0.000301,3,0
3,saint_supervised,ft_transformer,roc_auc,3,0.000519,0.000170,3,0


,reference_model,competitor_model,metric,n_paired_seeds,mean_reference_advantage,std_reference_advantage,reference_wins,ties
0,saint_supervised,baseline_ridge,rmse,3,0.236147,0.003245,3,0
1,saint_supervised,tabnet,rmse,3,0.191583,0.022213,3,0
2,saint_supervised,tab_transformer,rmse,3,0.030048,0.001193,3,0
3,saint_supervised,ft_transformer,rmse,3,0.016012,0.007702,3,0


## 10. Visualización de rendimiento y estabilidad

Los paneles mantienen escalas separadas. Las barras de error representan una desviación estándar entre semillas, no un intervalo de confianza. El mapa de rangos permite distinguir una ventaja consistente de un promedio producido por cambios de orden.

In [8]:
if COMPARISON_READY:
    performance_figure = plot_cross_task_performance(
        classification_summary,
        regression_summary,
        FIGURES_DIR,
    )
    rank_figure = plot_seed_rank_consistency(
        classification_ranks,
        regression_ranks,
        FIGURES_DIR,
    )
    display(
        pd.Series(
            {
                "performance": performance_figure.relative_to(PROJECT_ROOT),
                "ranks": rank_figure.relative_to(PROJECT_ROOT),
            }
        )
    )
else:
    display(pd.Series({"status": "cross-task figures pending"}))

performance    results\figures\comparison_task_performance.png
ranks          results\figures\comparison_seed_ranks.png
dtype: object

## 11. Efecto relativo frente al baseline

Para clasificación se informa diferencia absoluta de ROC-AUC respecto a Logistic Regression. Para regresión se informa reducción porcentual de RMSE respecto a Ridge. Estas dos columnas responden a preguntas análogas dentro de cada tarea, pero no forman una métrica universal ni deben sumarse.

In [9]:
if classification_artifacts.ready:
    display(
        classification_summary[[
            "model_name",
            "primary_effect_vs_baseline",
            "effect_unit",
        ]]
    )
if regression_artifacts.ready:
    display(
        regression_summary[[
            "model_name",
            "primary_effect_vs_baseline",
            "effect_unit",
        ]]
    )

,model_name,primary_effect_vs_baseline,effect_unit
0,saint_supervised,0.068148,absolute ROC-AUC difference
1,ft_transformer,0.067629,absolute ROC-AUC difference
2,tab_transformer,0.067108,absolute ROC-AUC difference
3,tabnet,0.062202,absolute ROC-AUC difference
4,baseline_logistic,0.000000,absolute ROC-AUC difference


,model_name,primary_effect_vs_baseline,effect_unit
0,saint_supervised,32.741007,RMSE reduction (%)
1,ft_transformer,30.520929,RMSE reduction (%)
2,tab_transformer,28.574905,RMSE reduction (%)
3,tabnet,6.178610,RMSE reduction (%)
4,baseline_ridge,0.000000,RMSE reduction (%)


## 12. Costo computacional y tamaño

El tiempo se resume dentro de cada tarea y hardware. La escala logarítmica evita que el baseline oculte diferencias entre redes. El número de parámetros mide capacidad y almacenamiento, pero no captura por sí solo memoria de activaciones, complejidad de atención ni costo del preprocesamiento.

In [10]:
if COMPARISON_READY:
    resource_figure = plot_cross_task_resources(
        classification_summary,
        regression_summary,
        FIGURES_DIR,
    )
    display(resource_figure.relative_to(PROJECT_ROOT))
else:
    display(pd.Series({"status": "resource comparison pending"}))

WindowsPath('results/figures/comparison_resources.png')

## 13. Early stopping y agotamiento del presupuesto

`budget_exhaustion_rate=1` significa que todas las semillas llegaron a la época máxima. No demuestra por sí solo subentrenamiento, pero indica que early stopping no cerró el proceso antes del límite y que la arquitectura pudo quedar condicionada por el presupuesto. Un valor menor revela que al menos una semilla restauró un checkpoint anterior.

In [11]:
budget_columns = [
    "model_name",
    "best_epoch_mean",
    "epochs_trained_mean",
    "budget_exhaustion_rate",
]
if classification_artifacts.ready:
    display(classification_summary[budget_columns])
if regression_artifacts.ready:
    display(regression_summary[budget_columns])

,model_name,best_epoch_mean,epochs_trained_mean,budget_exhaustion_rate
0,saint_supervised,36.333333,40.0,1.0
1,ft_transformer,39.000000,40.0,1.0
2,tab_transformer,39.000000,40.0,1.0
3,tabnet,38.666667,40.0,1.0
4,baseline_logistic,1.000000,1.0,NaN


,model_name,best_epoch_mean,epochs_trained_mean,budget_exhaustion_rate
0,saint_supervised,53.333333,59.333333,0.666667
1,ft_transformer,50.333333,57.000000,0.666667
2,tab_transformer,59.333333,60.000000,1.000000
3,tabnet,57.666667,60.000000,1.000000
4,baseline_ridge,1.000000,1.000000,NaN


## 14. Síntesis cruzada de arquitecturas

La tabla siguiente alinea exclusivamente las cuatro redes profundas. Conserva por separado ROC-AUC y RMSE, sus dispersiones, efectos frente al baseline, tiempos y parámetros. `descriptive_mean_rank` promedia los rangos internos entre esas cuatro redes; los rangos globales, que sí incluyen el baseline, se conservan como contexto. Este resumen ordinal facilita la lectura, pero no convierte problemas distintos en una única función objetivo.

In [12]:
architecture_table = (
    build_cross_task_architecture_table(
        classification_summary,
        regression_summary,
    )
    if COMPARISON_READY
    else pd.DataFrame()
)
if COMPARISON_READY:
    display(architecture_table)
else:
    display(pd.Series({"status": "architecture synthesis pending"}))

,model_name,classification_overall_rank,classification_roc_auc_mean,classification_roc_auc_std,classification_balanced_accuracy_mean,classification_auc_gain_vs_baseline,classification_train_time_mean,classification_parameters,classification_deep_rank,regression_overall_rank,regression_rmse_mean,regression_rmse_std,regression_mae_mean,regression_r2_mean,regression_rmse_reduction_percent,regression_train_time_mean,regression_parameters,regression_deep_rank,descriptive_mean_rank,rank_gap
0,saint_supervised,1,0.995818,0.000185,0.964595,0.068148,192.857010,59490,1,1,0.485111,0.003245,0.330029,0.822024,32.741007,36.500813,47233,1,1.0,0
1,ft_transformer,2,0.995298,0.000092,0.963153,0.067629,177.849243,27394,2,2,0.501123,0.006163,0.340173,0.810067,30.520929,26.205156,26049,2,2.0,0
2,tab_transformer,3,0.994777,0.000152,0.960971,0.067108,106.200336,53282,3,3,0.515159,0.004412,0.357733,0.799289,28.574905,15.208926,9473,3,3.0,0
3,tabnet,4,0.989872,0.000184,0.948178,0.062202,401.652661,62000,4,4,0.676694,0.025412,0.461130,0.653375,6.178610,108.214363,26224,4,4.0,0


## 15. Lectura arquitectónica

| Arquitectura | Mecanismo evaluado | Interpretación permitida | Precaución |
|---|---|---|---|
| TabNet | Máscaras secuenciales y pasos de decisión | Sensibilidad de selección condicional de columnas | Las máscaras no son evidencia causal y el entrenamiento puede ser sensible al presupuesto |
| TabTransformer | Atención sobre categorías y MLP numérico | Comportamiento con categorías contextualizadas en Airlines | California Housing no tiene categorías; allí se evalúa principalmente su rama numérica |
| FT-Transformer | Tokenización simétrica de variables numéricas y categóricas | Capacidad para modelar interacciones globales entre columnas | Puede pagar costo de atención aunque la tarea no requiera todas esas interacciones |
| SAINT inductivo | Embeddings no lineales y atención entre columnas | Interacciones intraobservación con predicciones independientes por fila | No se evalúa el posible beneficio transductivo de la atención entre filas |

## 16. Factores específicos de cada dataset

Airlines combina categorías, variables continuas y puntuaciones ordinales posteriores a la experiencia. El resultado describe satisfacción observada; no equivale necesariamente a una predicción disponible antes del vuelo.

California Housing es completamente numérico y tiene estructura espacial. La división aleatoria mide principalmente interpolación entre bloques mezclados geográficamente; no demuestra extrapolación a regiones nuevas. Además, el target presenta un techo conocido que afecta errores en viviendas de alto valor.

## 17. Qué no puede concluirse

- Tres semillas son suficientes para detectar inestabilidad evidente, pero no para estimar con precisión una distribución de rendimiento.
- Un rango promedio no prueba superioridad universal ni intercambiabilidad entre tareas.
- El mejor resultado de test no autoriza una nueva ronda de ajuste sobre los mismos splits.
- La comparación no aísla cada componente mediante ablations; contrasta configuraciones base coherentes de familias completas.
- El costo observado pertenece a este equipo, estas versiones y estos presupuestos.

## 18. Conclusiones derivadas de la ejecución

La celda final informa únicamente hechos calculados: mejor media dentro de cada tarea, dispersión, consistencia de victorias pareadas y posición cruzada descriptiva. Si falta una campaña, no reemplaza evidencia con texto predeterminado.

In [13]:
if COMPARISON_READY:
    class_best = classification_summary.iloc[0]
    regression_best = regression_summary.iloc[0]
    cross_task_first = architecture_table.iloc[0]
    class_closest = classification_differences.sort_values(
        "mean_reference_advantage"
    ).iloc[0]
    regression_closest = regression_differences.sort_values(
        "mean_reference_advantage"
    ).iloc[0]
    conclusions = pd.Series(
        {
            "classification_best_mean_model": class_best["model_name"],
            "classification_roc_auc_mean": class_best["roc_auc_mean"],
            "classification_roc_auc_std": class_best["roc_auc_std"],
            "classification_closest_competitor": class_closest[
                "competitor_model"
            ],
            "classification_advantage_over_closest": class_closest[
                "mean_reference_advantage"
            ],
            "classification_seed_wins_over_closest": (
                f"{int(class_closest['reference_wins'])}/"
                f"{int(class_closest['n_paired_seeds'])}"
            ),
            "regression_best_mean_model": regression_best["model_name"],
            "regression_rmse_mean": regression_best["rmse_mean"],
            "regression_rmse_std": regression_best["rmse_std"],
            "regression_closest_competitor": regression_closest[
                "competitor_model"
            ],
            "regression_advantage_over_closest": regression_closest[
                "mean_reference_advantage"
            ],
            "regression_seed_wins_over_closest": (
                f"{int(regression_closest['reference_wins'])}/"
                f"{int(regression_closest['n_paired_seeds'])}"
            ),
            "lowest_descriptive_mean_rank": cross_task_first["model_name"],
            "identical_deep_order_across_tasks": bool(
                architecture_table["rank_gap"].eq(0).all()
            ),
            "classification_protocol": classification_artifacts.metrics[
                "protocol_fingerprint"
            ].iloc[0],
            "regression_protocol": regression_artifacts.metrics[
                "protocol_fingerprint"
            ].iloc[0],
        },
        name="empirical_conclusions",
    )
    display(conclusions)
else:
    display(
        pd.Series(
            {
                "status": "comparison incomplete",
                "classification_ready": classification_artifacts.ready,
                "regression_ready": regression_artifacts.ready,
            },
            name="empirical_conclusions",
        )
    )

classification_best_mean_model                                            saint_supervised
classification_roc_auc_mean                                                       0.995818
classification_roc_auc_std                                                        0.000185
classification_closest_competitor                                           ft_transformer
classification_advantage_over_closest                                             0.000519
classification_seed_wins_over_closest                                                  3/3
regression_best_mean_model                                                saint_supervised
regression_rmse_mean                                                              0.485111
regression_rmse_std                                                               0.003245
regression_closest_competitor                                               ft_transformer
regression_advantage_over_closest                                                 0.016012